# QuantSupport Python bindings — a guided tour

This notebook walks through every component of the `quantsupport` Python API:

1. **Dates, periods and calendars** — the time toolkit
2. **Enums** — currencies, market indices, conventions
3. **Market data** — quote stores, curve configurations, FX
4. **The pricing context** — bootstrapping and lifecycle
5. **Exploring bootstrapped curves** — nodes, discount factors, forward rates, pillars
6. **Pricing trades** — swaps and cross-currency swaps with AD sensitivities
7. **XVA** — per-client CSA terms, netting sets, Monte Carlo exposures

Build and install the bindings first (from the repo root):

```bash
pip install maturin
maturin develop --release -m bindings/python/Cargo.toml
```

In [2]:
from pathlib import Path

import quantsupport as qs

# Market data JSON files shipped with the Rust CVA example.
DATA = next(
    p / "examples" / "cva" / "data"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "examples" / "cva" / "data").exists()
)
DATA

ModuleNotFoundError: No module named 'quantsupport'

## 1. Dates, periods and calendars

`qs.Date` is an immutable calendar date. It supports arithmetic with integers
(days), `qs.Period` objects, or period strings like `"6M"` / `"1Y6M"`.
Subtracting two dates gives the number of days between them.

In [ ]:
d = qs.Date(2025, 11, 11)          # or qs.Date.parse("2025-11-11")
print(d, "is a", d.weekday())
print("+30 days:      ", d + 30)
print("+6 months:     ", d + "6M")
print("+1 year 6 mo:  ", d + qs.Period.parse("1Y6M"))
print("end of month:  ", d.end_of_month())
print("days to +6M:   ", (d + "6M") - d)

In [ ]:
p = qs.Period(18, qs.TimeUnit.Months)
print(p, "->", p.length, p.units)
print("from frequency:", qs.Period.from_frequency(qs.Frequency.Quarterly))

`qs.Calendar` answers business-day questions: adjusting dates with a
`BusinessDayConvention`, advancing by periods while skipping holidays, and
listing holidays or business days. `qs.DayCounter` measures year fractions
between dates under a market convention.

In [ ]:
cal = qs.Calendar("UnitedStates")   # also: TARGET, Brazil, Chile, WeekendsOnly, NullCalendar
thanksgiving = qs.Date(2025, 11, 27)
print("business day?          ", cal.is_business_day(thanksgiving))
print("adjusted (Following):  ", cal.adjust(thanksgiving, qs.BusinessDayConvention.Following))
print("advance 5 business d.: ", cal.advance(d, "5D"))
print("business days in Nov:  ", cal.business_days_between(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))
print("holidays in Nov 2025:  ", cal.holiday_list(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))

dc = qs.DayCounter.Actual360
print("Act/360 year fraction: ", dc.year_fraction(d, d + "6M"))

## 2. Enums

All conventions are proper Python enum classes (hashable, comparable, usable as
dict keys). Every API that takes an enum also accepts its string name, and each
enum has a case-insensitive `parse` static method.

`qs.Currency` carries full ISO 4217 metadata:

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        "code": c.code,
        "name": c.name,
        "symbol": c.symbol,
        "precision": c.precision,
        "numeric_code": c.numeric_code,
    }
    for c in [qs.Currency.USD, qs.Currency.EUR, qs.Currency.CLP, qs.Currency.JPY, qs.Currency.BRL]
)

`qs.MarketIndex` identifies every curve, surface and simulation: overnight
rates (`SOFR`, `ESTR`, `ICP`, ...), term rates (`EURIBOR6m`, `TermSOFR3m`, ...),
plus parameterised indices built with static factories.

In [ ]:
print(qs.MarketIndex.SOFR, "|", qs.MarketIndex.EURIBOR6m, "|", qs.MarketIndex.ICP)
print("equity:    ", qs.MarketIndex.equity("AAPL"))
print("fx pair:   ", qs.MarketIndex.fx_pair("CLP", "USD"))
print("collateral:", qs.MarketIndex.collateral("CLP", "USD"))
print("parse:     ", qs.MarketIndex.parse("sofr"))

# Other conventions follow the same pattern:
print(qs.Side.LongReceive.sign(), qs.Side.parse("pay"))
print(qs.Compounding.Compounded, qs.Frequency.Semiannual, qs.Request.Sensitivities)

## 3. Market data

A `qs.QuoteStore` holds all market quotes as of a single reference date. It
deserializes straight from JSON and can be inspected as a DataFrame.

In [ ]:
quotes = qs.QuoteStore.from_json(str(DATA / "quotes.json"))
print("reference date:", quotes.reference_date, "| quotes:", len(quotes.identifiers()))
quotes.to_dataframe().head(8)

`qs.CurveConfiguration` describes how each curve is bootstrapped: which market
index it belongs to, its day counter and interpolator, and which quotes pin its
pillars. `to_dict()` exposes the full configuration for inspection.

In [ ]:
curves = [
    c for c in qs.CurveConfiguration.from_json(str(DATA / "curve_specs.json"))
    if "ESTR" not in repr(c)   # the sample quote file has no ESTR quotes
]
for c in curves:
    cfg = c.to_dict()
    print(f"{cfg['market_index']!s:<22} {cfg['day_counter']:<10} "
          f"{cfg['interpolator']:<18} {len(cfg['quotes'])} quotes")

In [ ]:
# FX spot rates and the base discounting configuration of the context.
fx = qs.FxStore.from_dict([{"base": "CLP", "quote": "USD", "rate": 1.0 / 900.0}])
discounting = qs.DiscountingConfig(currency=qs.Currency.USD, index=qs.MarketIndex.SOFR)
discounting

## 4. The pricing context

`qs.PricingContext` bundles all market data. **Initializing** it bootstraps
every configured curve, builds volatility surfaces/cubes and runs any
configured simulations.

There are two ways to drive its lifecycle:

- `ctx.initialize()` — bootstrap once, then explore the constructed objects
  freely across notebook cells (used in section 5);
- `with ctx: ...` — additionally starts the automatic-differentiation tape,
  which is required for pricing with sensitivities, and releases everything on
  exit (used in sections 6–7).

In [ ]:
ctx = qs.PricingContext(quotes=quotes, curves=curves, fx=fx, discounting=discounting)
ctx.initialize()

ref = ctx.reference_date            # a qs.Date
print("reference date:", ref)
print("original curve configurations:", [str(c.to_dict()['market_index']) for c in ctx.curve_configurations])
print("original quotes:", len(ctx.quotes.identifiers()))

## 5. Exploring bootstrapped curves

After initialization the context holds the *constructed* market data. Each
bootstrapped curve is exposed as a `qs.DiscountCurve` view with:

- `nodes()` — the raw date/discount-factor nodes as a DataFrame,
- `discount_factor(date)` and `forward_rate(start, end, compounding, frequency)`,
- `pillars()` — the quote pillars the bootstrap was calibrated to.

(The same pattern applies to `ctx.volatility_surfaces()`, `ctx.volatility_cubes()`
and `ctx.simulations()` when those are configured.)

In [ ]:
for curve in ctx.curves():
    print(f"{curve.market_index!s:<22} ref={curve.reference_date}  "
          f"day_counter={curve.day_counter}  nodes={len(curve.nodes())}")

In [ ]:
sofr = ctx.curve(qs.MarketIndex.SOFR)
sofr.nodes().head(6)

In [ ]:
# Discount factors and forward rates at arbitrary dates.
term = pd.DataFrame(
    {
        "tenor": t,
        "discount_factor": sofr.discount_factor(ref + t),
        "fwd_1y": sofr.forward_rate(ref + t, (ref + t) + "1Y",
                                    qs.Compounding.Simple, qs.Frequency.Annual),
    }
    for t in ["1Y", "2Y", "5Y", "10Y", "20Y"]
)
term

In [ ]:
# The quote pillars the curve was calibrated to (bootstrap inputs).
sofr.pillars()

## 6. Pricing trades

Trades are built with typed enums throughout. `ctx.evaluate` takes a list of
`qs.Request` values; sensitivities are computed by automatic differentiation
with respect to the original market quotes, so pricing runs inside a
`with ctx:` block (which starts the AD tape).

In [ ]:
swap = qs.Swap(
    identifier="USD_IRS_5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=10_000_000.0,
    fixed_rate=0.0378,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    side=qs.Side.LongReceive,
    fixed_leg_frequency=qs.Frequency.Quarterly,
    floating_leg_frequency=qs.Frequency.Semiannual,
)
print(swap.identifier, "|", swap.currency, swap.market_index, swap.side,
      "|", swap.start_date, "->", swap.maturity_date)

with ctx:
    res = ctx.evaluate(swap, [qs.Request.Value, qs.Request.Cashflows, qs.Request.Sensitivities])
    cashflows = res.cashflows
    sens = res.sensitivities

print(f"NPV = {res.price:,.2f}")
cashflows.head(6)

In [ ]:
# Quote-level sensitivities (per unit move of each input quote), largest first.
sens.reindex(sens.value.abs().sort_values(ascending=False).index).head(8)

In [ ]:
xccy = qs.CrossCurrencySwap(
    identifier="CLPUSD_XCCY_5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    domestic_notional=10_000_000.0,
    foreign_notional=10_000_000.0 * 900.0,
    domestic_currency=qs.Currency.USD,
    foreign_currency=qs.Currency.CLP,
    domestic_market_index=qs.MarketIndex.SOFR,
    foreign_market_index=qs.MarketIndex.ICP,
    foreign_spread=0.002,
    side=qs.Side.LongReceive,
)

with ctx:
    xres = ctx.evaluate(xccy, [qs.Request.Value])
print(f"{xccy.identifier} NPV = {xres.price:,.2f}")

## 7. XVA — per-client CSA terms and Monte Carlo exposures

The XVA engine simulates portfolio values along Monte Carlo paths and
aggregates CVA/FVA per **netting set**. Each netting set (one client)
carries its own `qs.CsaTerms`:

- `collateral_index` / `collateral_currency` — which curve remunerates posted
  collateral (drives CSA discounting),
- `credit_spread`, `recovery` — the client's credit parameters for CVA,
- `funding_spread` — for FVA,
- optional `credit_index` — a bootstrapped credit curve to use instead of the
  flat spread.

`qs.XvaConfig` holds the simulation setup (paths, seed, model configurations).

In [ ]:
xva_config = qs.XvaConfig.from_json(str(DATA / "xva_config.json"))
csa = qs.CsaTerms.from_json(str(DATA / "csa_terms.json"))
print(xva_config)
print(csa)

# CSA terms can also be built inline, per client:
csa_client_b = qs.CsaTerms(
    collateral_index=qs.MarketIndex.SOFR,
    collateral_currency=qs.Currency.USD,
    credit_spread=0.02,
    recovery=0.4,
    funding_spread=0.005,
)

In [ ]:
with ctx:
    result = ctx.run_xva(
        xva_config,
        netting_sets=[
            qs.NettingSet("client_a", [swap, xccy], csa),
            qs.NettingSet("client_b", [swap], csa_client_b),
        ],
    )
    xva_values = result.xva_values
    xva_sens = result.sensitivities
    exposures = result.exposures

xva_values

In [ ]:
# Expected positive/negative exposure profiles per netting set.
profile = exposures[0]
print(profile)
profile.to_dataframe().head(8)

In [ ]:
# XVA sensitivities: market-quote deltas plus per-client credit/funding parameters.
xva_sens.reindex(xva_sens.value.abs().sort_values(ascending=False).index).head(10)

## Wrap-up

- Every convention is a typed enum (`Currency`, `MarketIndex`, `Side`,
  `Compounding`, `Frequency`, `DayCounter`, `BusinessDayConvention`,
  `Request`, ...); strings are accepted everywhere as a convenience.
- `Date` / `Period` / `Calendar` cover schedule and business-day arithmetic.
- The `PricingContext` exposes both its **inputs** (`quotes`,
  `curve_configurations`, ...) and its **constructed outputs** (`curves()`,
  `volatility_surfaces()`, `volatility_cubes()`, `simulations()`).
- All tabular results (`nodes`, `pillars`, `cashflows`, `sensitivities`,
  `xva_values`, exposure profiles) are pandas DataFrames.